# Vietnamese BF+RAG Experiments — Kaggle Runner

**Before running:**
1. Settings → Accelerator → **GPU T4 x1** (or P100)
2. Settings → Internet → **On**
3. Add your HuggingFace token: Settings → Secrets → Add `HF_TOKEN`

**Run order:** Cell 1 (setup) → Cell 2 (clone) → Cell 3 (install) → Cell 4 (auth) → Cell 5 (validate) → Cell 6 (RAG index) → Cell 7 (smoke run) → Cell 8 (summarize)

In [ ]:
# Cell 1 — Check GPU and environment
import subprocess, os, sys

print('=== GPU ===')
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'], check=False)

import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

WORK_DIR = '/kaggle/working'
print(f'\nWorking dir: {WORK_DIR}')
print(f'Disk free: ', end='')
subprocess.run(['df', '-h', WORK_DIR])

In [ ]:
# Cell 2 — Clone repo
import os

REPO_DIR = '/kaggle/working/s1_budget_forcing'

if not os.path.exists(REPO_DIR):
    # Option A: clone from GitHub (if repo is public)
    # !git clone https://github.com/YOUR_USERNAME/s1_budget_forcing.git {REPO_DIR}

    # Option B: upload repo as a Kaggle Dataset and copy
    # The dataset would be at /kaggle/input/s1-budget-forcing/
    import shutil
    INPUT_PATH = '/kaggle/input/s1-budget-forcing'
    if os.path.exists(INPUT_PATH):
        shutil.copytree(INPUT_PATH, REPO_DIR)
        print(f'Copied from Kaggle dataset to {REPO_DIR}')
    else:
        raise FileNotFoundError(
            'Repo not found. Either:\n'
            '  A) Uncomment the git clone line above, or\n'
            '  B) Upload the repo as a Kaggle dataset named "s1-budget-forcing"'
        )
else:
    print(f'Repo already at {REPO_DIR}')

os.chdir(REPO_DIR)
print(f'CWD: {os.getcwd()}')
!ls

In [ ]:
# Cell 3 — Install dependencies
# Kaggle has torch/transformers pre-installed; only install what's missing.
!pip install -q \
    faiss-cpu>=1.7.4 \
    sentence-transformers>=2.7.0 \
    datasets>=2.18.0 \
    bitsandbytes>=0.43.0 \
    accelerate>=0.27.0 \
    tqdm

print('\nInstallation complete.')

In [ ]:
# Cell 4 — HuggingFace authentication
# Required for gated models (Vinallama, Vistral may need it).
# Store your token in Kaggle Secrets as HF_TOKEN — never hardcode it here.
from kaggle_secrets import UserSecretsClient
import os

try:
    secret = UserSecretsClient()
    hf_token = secret.get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print('HuggingFace login successful.')
except Exception as e:
    print(f'[WARN] HF auth skipped: {e}')
    print('Public models (Qwen2.5, SeaLLM) will still work without a token.')

In [ ]:
# Cell 5 — Validate Vietnamese benchmarks
import sys, os
sys.path.insert(0, '/kaggle/working/s1_budget_forcing/experiments')
os.chdir('/kaggle/working/s1_budget_forcing')

!python experiments/data/download_vi_benchmarks.py --n_samples 2

In [ ]:
# Cell 6 — Build Vietnamese Wikipedia RAG index
#
# Option A (smoke/fast): 10k articles, ~5 min on CPU
# Option B (full): all ~1.5M articles, ~60-90 min — save as Kaggle dataset to reuse
#
# For first run use Option A. Once confirmed, run Option B and save to /kaggle/working/vi_wiki_index_full/
# then upload that folder as a Kaggle dataset to skip this step in future sessions.

import os

SMOKE_INDEX = '/kaggle/working/vi_wiki_index_smoke'
FULL_INDEX  = '/kaggle/input/vi-wiki-index'  # pre-built dataset (if available)

if os.path.exists(FULL_INDEX + '/index.faiss'):
    print(f'Pre-built full index found at {FULL_INDEX} — skipping build.')
    RAG_INDEX_DIR = FULL_INDEX
elif not os.path.exists(SMOKE_INDEX + '/index.faiss'):
    print('Building smoke index (10k articles)...')
    !python experiments/rag/knowledge_base.py \
        --output_dir {SMOKE_INDEX} \
        --max_docs 10000 \
        --batch_size 512
    RAG_INDEX_DIR = SMOKE_INDEX
else:
    print(f'Smoke index already exists at {SMOKE_INDEX}')
    RAG_INDEX_DIR = SMOKE_INDEX

print(f'\nUsing RAG index: {RAG_INDEX_DIR}')

In [ ]:
# Cell 7 — Run experiments
#
# Edit the variables below to control the sweep.
# Start with SMOKE settings, then switch to SMALL_MATRIX after confirming output shape.

import os, subprocess
os.chdir('/kaggle/working/s1_budget_forcing')

# ── Choose a preset ─────────────────────────────────────────────────────────
PRESET = 'SMOKE'   # 'SMOKE' | 'SMALL_MATRIX' | 'FULL'

PRESETS = {
    'SMOKE': dict(
        MODELS       = 'qwen2.5-3B',
        BENCHMARKS   = 'vi_gsm8k',
        CONDITIONS   = 'BF_only RAG_only BF_RAG',
        N_WAIT_LIST  = '0 1 2',
        N_SAMPLES    = '5',
        EXTRA_ARGS   = '--max_tokens 512',   # 4-bit ON by default on Kaggle GPU
    ),
    'SMALL_MATRIX': dict(
        MODELS       = 'qwen2.5-3B qwen2.5-7B',
        BENCHMARKS   = 'vi_gsm8k vimmlu',
        CONDITIONS   = 'BF_only RAG_only BF_RAG',
        N_WAIT_LIST  = '0 1 2',
        N_SAMPLES    = '20',
        EXTRA_ARGS   = '',
    ),
    'FULL': dict(
        MODELS       = 'qwen2.5-3B qwen2.5-7B vinallama-7b vistral-7b seallm-7b',
        BENCHMARKS   = 'vi_gsm8k vimmlu',
        CONDITIONS   = 'BF_only RAG_only BF_RAG',
        N_WAIT_LIST  = '0 1 2',
        N_SAMPLES    = '100',
        EXTRA_ARGS   = '',
    ),
}

cfg = PRESETS[PRESET]
print(f'Running preset: {PRESET}')
for k, v in cfg.items():
    print(f'  {k}={v!r}')
print(f'  RAG_INDEX_DIR={RAG_INDEX_DIR!r}')

env = os.environ.copy()
env.update({
    'MODELS':       cfg['MODELS'],
    'BENCHMARKS':   cfg['BENCHMARKS'],
    'CONDITIONS':   cfg['CONDITIONS'],
    'N_WAIT_LIST':  cfg['N_WAIT_LIST'],
    'N_SAMPLES':    cfg['N_SAMPLES'],
    'OUTPUT_DIR':   'experiments/results',
    'RAG_INDEX_DIR': RAG_INDEX_DIR,
    'EXTRA_ARGS':   cfg['EXTRA_ARGS'],
})

result = subprocess.run(
    ['bash', 'experiments/scripts/run_vi_bf_rag.sh'],
    env=env,
    cwd='/kaggle/working/s1_budget_forcing',
)
print(f'\nExit code: {result.returncode}')

In [ ]:
# Cell 8 — Aggregate results
import glob, os

results_root = '/kaggle/working/s1_budget_forcing/experiments/results'
run_dirs = sorted(glob.glob(f'{results_root}/vi_*'))

if not run_dirs:
    print('No vi_* result directories found. Run Cell 7 first.')
else:
    latest = run_dirs[-1]
    print(f'Aggregating: {latest}')
    !python experiments/results/summary_vi.py --results_dir {latest}

    # Print the CSV
    import pandas as pd
    csv_path = os.path.join(latest, 'summary_vi.csv')
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        display_cols = ['model', 'benchmark', 'condition', 'n_wait', 'n_samples',
                        'accuracy', 'scaling', 'performance', 'extraction_failures']
        print(df[display_cols].to_string(index=False))
    else:
        print(f'CSV not found at {csv_path}')

In [ ]:
# Cell 9 — Copy outputs to /kaggle/working for download
# Kaggle allows downloading files from /kaggle/working/.
import shutil, os, glob

OUT = '/kaggle/working/outputs'
os.makedirs(OUT, exist_ok=True)

# Copy all result directories
for d in glob.glob('/kaggle/working/s1_budget_forcing/experiments/results/vi_*'):
    dest = os.path.join(OUT, os.path.basename(d))
    if not os.path.exists(dest):
        shutil.copytree(d, dest)
        print(f'Copied: {dest}')

print(f'\nAll outputs in: {OUT}')
!ls -lh {OUT}